In [ ]:
# Node LTS and Mermaid CLI (Skip when need to only train)
# !apt-get update -qq && apt-get install -y chromium-browser libnss3 libgconf-2-4 libfontconfig1 -qq
# !npm install -g @mermaid-js/mermaid-cli --silent

#!uv init .
#!uv add --dev ipykernel
#!uv add "transformers>=4.49.0"

!npm install @mermaid-js/mermaid-cli
!uv pip install -q ai-edge-torch-nightly
!uv pip install -q --upgrade "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --no-deps
!uv pip install -q unsloth_zoo
# !uv pip install -U --no-cache-dir unsloth transformers trl unsloth_zoo

In [ ]:
!hf auth login   # if the dataset needs a token
!hf download colinfrisch/diagrams-mermaid-filtered \
  --repo-type dataset \
  --local-dir ./data/diagrams-mermaid-filtered

In [ ]:
from datasets import load_dataset
local_path = "./data/diagrams-mermaid-filtered"  # or absolute path
dataset = load_dataset("parquet", data_dir=local_path, split="train")
split_dataset = dataset.train_test_split(test_size=1000, seed=42)
eval_dataset = split_dataset["test"]

In [ ]:
# ===========================
# Validate on macOS (Apple Silicon) + Mermaid CLI
# ===========================
import json
import os
import subprocess
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "./output/gemma-3-1b-mermaid/gemma-mermaid-merged"

# True: print raw completion, extracted ```mermaid``` block, and dataset reference text.
DEBUG_VALIDATE = True

# Prefer MPS on M2; CPU is the reliable fallback if MPS errors.
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# On some transformers builds, passing fix_mistral_regex raises TypeError
# (duplicate kwarg inside _patch_mistral_regex). Omit until you upgrade HF.
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype=torch.float16,  # was torch_dtype=torch.float16
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.eval()
model.to(device)


def validate_mermaid(mmd_code, filename="test.mmd"):
    """mmdc on macOS: usually no Linux/Docker puppeteer sandbox flags needed."""
    with open(filename, "w", encoding="utf-8") as f:
        f.write(mmd_code)
    try:
        result = subprocess.run(
            ["npx", "--no-install", "mmdc", "-i", filename, "-o", "test.svg"],
            cwd=os.getcwd(),
            capture_output=True,
            text=True,
            timeout=60,
        )
        return result.returncode == 0, (result.stdout or "") + (result.stderr or "")
    except Exception as e:
        return False, str(e)


# Only if you loaded the model with Unsloth — omit for plain AutoModelForCausalLM above.
# FastLanguageModel.for_inference(model)

n = min(20, len(eval_dataset))
success_count = 0
test_samples = eval_dataset.select(range(n))

for idx, example in enumerate(test_samples):
    try:
        messages = example["messages"]
        user_content = messages[0]["content"]

        prompt = tokenizer.apply_chat_template(
            [{"role": "user", "content": user_content}],
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = tokenizer(prompt, return_tensors="pt")
        input_ids = inputs.input_ids.to(device)
        attention_mask = inputs.attention_mask.to(device)

        with torch.no_grad():
            outputs = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=512,
                use_cache=True,  # if MPS errors: try False
                temperature=0.2,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )

        input_length = input_ids.shape[1]
        generated = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

        ref_assistant = None
        for m in messages:
            if m.get("role") in ("assistant", "model"):
                ref_assistant = m.get("content", "")
                break
        if DEBUG_VALIDATE:
            print(f"   [debug] completion len={len(generated)}")
            print(generated[:1500])
            if ref_assistant is not None:
                print(f"   [debug] reference assistant len={len(ref_assistant)}")
                print(ref_assistant[:800])

        mmd = None
        if "```mermaid" in generated:
            start = generated.find("```mermaid") + len("```mermaid")
            end = generated.find("```", start)
            mmd = generated[start:end].strip()

        if mmd:
            valid, log = validate_mermaid(mmd)
            success_count += int(valid)
            print(f"Sample {idx + 1}/{n} | Valid: {valid}")
            if not valid:
                log_show = log if DEBUG_VALIDATE else log[:500]
                print(f"   Error log: {log_show}")
                if DEBUG_VALIDATE:
                    print(f"   Extracted mermaid ({len(mmd)} chars):\n{mmd[:1200]}")
        else:
            print(f"Sample {idx + 1}/{n} | No Mermaid block found")
            if DEBUG_VALIDATE and generated:
                print(f"   [debug] raw tail: {generated[-400:]}")
    except Exception as e:
        print(f"Sample {idx + 1}/{n} | Error: {e}")

pct = (success_count / n * 100) if n else 0.0
print(f"\n✅ Mermaid compilation success rate: {success_count}/{n} = {pct:.1f}%")